# Whitening

A gravitational-wave detector is an extraordinarily sensitive microphone for spacetime.
But like any microphone, it does not hear all frequencies equally — some are deafeningly
loud, others barely a whisper.

**Whitening** is the process of equalising the volume across all frequencies, making a
signal that was invisible in loud noise stand out clearly.

This notebook builds that intuition step by step, with plots you can see and sounds you
can hear — no prior signal-processing background needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import welch
from IPython.display import Audio, display

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110})

FS       = 22050   # audio sample rate (Hz)
DURATION = 5.0     # seconds
N        = int(FS * DURATION)
t        = np.linspace(0, DURATION, N, endpoint=False)

def to_audio(x):
    return (x / (np.max(np.abs(x)) + 1e-10) * 0.9).astype(np.float32)

def specgram(ax, x, title, vmin=-80, vmax=0):
    f, ts, Sxx = signal.spectrogram(x, fs=FS, nperseg=1024, noverlap=768, scaling='spectrum')
    db = 10 * np.log10(np.maximum(Sxx, 1e-12))
    ax.pcolormesh(ts, f, db, vmin=vmin, vmax=vmax, cmap='inferno', shading='gouraud')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(title)

print(f'{N:,} samples  |  {FS} Hz  |  {DURATION:.0f} s')

---
## 1. What Is a Frequency?

Every signal — sound, electrical noise, gravitational waves — can be broken down into
**sine waves** oscillating at different rates. The rate is the **frequency**, measured in
**Hz** (cycles per second).

- **Low frequency** → slow oscillation → bass / rumble
- **High frequency** → fast oscillation → bright treble / hiss

Below: four sinusoids at very different frequencies, all plotted over the same 50 ms window.
Notice that low-frequency waves complete only a fraction of a cycle while high-frequency
waves pack in many cycles in the same time.

In [ ]:
t_ms = np.linspace(0, 50e-3, int(FS * 50e-3), endpoint=False)

freq_info = [
    (60,   'C0', '60 Hz — deep bass (mains hum)'),
    (440,  'C1', '440 Hz — middle A (musical tuning note)'),
    (2000, 'C2', '2 000 Hz — bright treble'),
    (8000, 'C3', '8 000 Hz — very high tone'),
]

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, (f, c, lbl) in zip(axes, freq_info):
    ax.plot(t_ms * 1e3, np.sin(2 * np.pi * f * t_ms), color=c, lw=1.5)
    ax.axhline(0, lw=0.5, color='k', alpha=0.3)
    ax.set_yticklabels([])
    ax.annotate(lbl, xy=(0.01, 0.75), xycoords='axes fraction',
                fontsize=9, color=c, fontweight='bold')

axes[-1].set_xlabel('Time (ms)')
fig.suptitle(
    'Four sinusoids at different frequencies (same 50 ms window)\n'
    'Low frequency = slow wiggle   |   High frequency = fast wiggle',
    fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Power and the Power Spectrum

### What is power?

The **power** at a given frequency is how much energy the signal carries there — it is
proportional to the *square* of the amplitude. Double the amplitude → four times the power.

A loud bass note carries high power at low frequencies.
A loud whistle carries high power at high frequencies.

### What is the power spectrum?

The **power spectrum** answers the question:
> *"At each frequency, how much power is there?"*

Think of it as a bar chart: x-axis is frequency, y-axis is loudness at that frequency.

Three examples below:
- **One tone** → one spike at its frequency, nothing elsewhere
- **Three tones** → three spikes, one per tone
- **White noise** → flat — *every* frequency equally loud (this is our target)

In [ ]:
t1 = np.linspace(0, 2.0, 2 * FS, endpoint=False)

sig_single = np.sin(2 * np.pi * 440 * t1)
f_s, p_s   = welch(sig_single, fs=FS, nperseg=FS // 2)

sig_three  = (np.sin(2 * np.pi * 300  * t1) +
              np.sin(2 * np.pi * 1000 * t1) +
              np.sin(2 * np.pi * 3000 * t1))
f_3, p_3   = welch(sig_three, fs=FS, nperseg=FS // 2)

wn_short       = np.random.randn(2 * FS)
f_wns, p_wns   = welch(wn_short, fs=FS, nperseg=FS // 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(f_s[1:], p_s[1:], lw=1.5)
axes[0].set_xlim(0, 5000); axes[0].set_ylim(1e-7, 10)
axes[0].axvline(440, color='red', ls='--', lw=1.5, alpha=0.8, label='440 Hz')
axes[0].set_title('Single tone (440 Hz)\none spike in the spectrum')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Power')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].semilogy(f_3[1:], p_3[1:], lw=1.5, color='C1')
axes[1].set_xlim(0, 5000); axes[1].set_ylim(1e-7, 10)
for fv, lbl in [(300, '300 Hz'), (1000, '1 000 Hz'), (3000, '3 000 Hz')]:
    axes[1].axvline(fv, color='red', ls='--', lw=1.5, alpha=0.8, label=lbl)
axes[1].set_title('Three tones (300, 1000, 3000 Hz)\nthree spikes')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

med = np.median(p_wns[1:])
axes[2].semilogy(f_wns[1:], p_wns[1:], lw=0.8, color='C2', alpha=0.9)
axes[2].axhline(med, color='red', ls='--', lw=2, label='Mean level')
axes[2].set_xlim(0, FS // 2); axes[2].set_ylim(1e-7, 10)
axes[2].set_title('White noise\nflat: every frequency equally loud')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

fig.suptitle('Power spectrum: the frequency "fingerprint" of a signal', fontsize=12)
plt.tight_layout(); plt.show()

---
## 3. White Noise — Equal Power at Every Frequency

**White noise** has a *flat* power spectrum: every frequency carries exactly the same power.

The name comes from **white light** — white light contains all colours of the visible spectrum
equally. White noise contains all audio frequencies equally. No single frequency sticks out.

Three ways to see this at once:

- **Time domain**: random jitter with no pattern and no preferred rhythm.
- **Power spectrum**: a flat horizontal line — no frequency is louder than any other.
- **Spectrogram**: uniformly bright at every frequency and every moment — no structure.

White noise is the *ideal* background for a detector: since no frequency stands out on its
own, any signal that does stand out is genuinely unusual.

In [ ]:
white_noise  = np.random.randn(N) * 0.3
f_wn, psd_wn = welch(white_noise, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t[:3000], white_noise[:3000], lw=0.5)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time domain\nRandom jitter — no pattern, no rhythm')

mean_db = 10 * np.log10(np.median(psd_wn[1:]))
axes[1].semilogx(f_wn[1:], 10 * np.log10(psd_wn[1:]), lw=1.2)
axes[1].axhline(mean_db, color='red', ls='--', lw=2, label='Mean level')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Power (dB)')
axes[1].set_title('Power spectrum\nFlat — all frequencies equal')
axes[1].set_xlim(20, FS // 2); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

f_sp, ts_sp, Sxx = signal.spectrogram(white_noise, fs=FS, nperseg=512, noverlap=384)
db_sp = 10 * np.log10(np.maximum(Sxx, 1e-12))
axes[2].pcolormesh(ts_sp, f_sp, db_sp, vmin=-50, vmax=10, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 5000)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('Spectrogram\nUniformly bright — no frequency dominates')

fig.suptitle('White noise: equal power at every frequency and every moment', fontsize=12)
plt.tight_layout(); plt.show()

print('White noise — uniform hiss, no tonal character. This is our target.')
display(Audio(to_audio(white_noise), rate=FS))

---
## 4. The Problem: Detector Noise Is Coloured

A real gravitational-wave detector's noise is the **opposite** of flat.

- **Below ~40 Hz**: seismic vibrations are enormous — the Earth shakes constantly and
  couples directly into the mirror suspensions.
- **Above ~1 kHz**: quantum shot noise from the laser takes over.
- **In between**: power-line harmonics (60 Hz and multiples), mirror resonances, and dozens
  of other instrumental artefacts add sharp spikes.

The power falls steeply with frequency — we call this **coloured noise**.

For our audio demo, we synthesise it: take white noise, multiply each Fourier coefficient
by $1/f^2$, then transform back. This gives the same steep spectral shape as a real
detector, instantly listenable.

The comparison plot is the key picture: the **gap** between the white and coloured lines is
the region where a gravitational-wave signal would be completely buried.

In [ ]:
raw         = np.random.randn(N)
raw_fd      = np.fft.rfft(raw)
freqs_fd    = np.fft.rfftfreq(N, 1.0 / FS)
freqs_fd[0] = 1.0   # avoid division by zero at DC

colour_filt = 1.0 / (np.maximum(freqs_fd, 1.0) / 60.0) ** 2
col_noise   = np.fft.irfft(raw_fd * colour_filt, n=N)
col_noise  /= np.std(col_noise)
col_noise  *= 0.65

f_c, psd_c = welch(col_noise, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Time domain comparison
scale_w = white_noise / np.std(white_noise)
scale_c = col_noise   / np.std(col_noise)
axes[0].plot(t[:4000], scale_w[:4000], lw=0.6, alpha=0.8, label='White noise')
axes[0].plot(t[:4000], scale_c[:4000], lw=0.7, alpha=0.8, color='C1', label='Coloured noise')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude (normalised)')
axes[0].set_title('Time domain\nColoured noise has large slow swings')
axes[0].legend(fontsize=9)

# Power spectrum comparison — the central picture
axes[1].loglog(f_wn[1:], np.sqrt(psd_wn[1:]), lw=2, color='C0', label='White (flat)')
axes[1].loglog(f_c[1:],  np.sqrt(psd_c[1:]),  lw=2, color='C1', label='Coloured (1/f²)')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Amplitude spectral density')
axes[1].set_title('Power spectra compared\nGap = where signals are buried')
axes[1].legend(fontsize=9); axes[1].grid(True, which='both', alpha=0.3)
axes[1].set_xlim(20, FS // 2)

# Spectrogram of coloured noise
f_sp2, ts_sp2, Sxx2 = signal.spectrogram(col_noise, fs=FS, nperseg=512, noverlap=384)
db_sp2 = 10 * np.log10(np.maximum(Sxx2, 1e-12))
axes[2].pcolormesh(ts_sp2, f_sp2, db_sp2, vmin=-30, vmax=30, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 5000)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('Coloured noise spectrogram\nBright at bottom, dark at top')

fig.suptitle('Coloured noise: low frequencies dominate', fontsize=12)
plt.tight_layout(); plt.show()

print('Coloured noise — heavy bass rumble. Low frequencies overwhelm everything else.')
display(Audio(to_audio(col_noise), rate=FS))

---
## 5. Whitening — Turning Coloured Into White

### The idea

For each frequency $f$, measure how loud the noise is there. That is the
**amplitude spectral density** $\sqrt{S_n(f)}$. Then divide:

$$\tilde{h}_\text{white}(f) = \frac{\tilde{h}(f)}{\sqrt{S_n(f)}}$$

- Where noise is **loud** (low $f$): divide by a **large** number → that frequency shrinks.
- Where noise is **quiet** (high $f$): divide by a **small** number → that frequency grows.

After dividing, every frequency ends up at the same level. Coloured → white.

**This is not a trick.** The signal-to-noise ratio at every frequency is exactly preserved —
you are *redistributing* the information, not discarding it.

### Three steps — shown in the plot below

Step 1 shows the raw steep spectrum. Step 2 shows the noise estimate used to divide.
Step 3 shows the result: flat.

In [ ]:
f_raw, psd_raw = welch(col_noise, fs=FS, nperseg=FS // 4)
asd_est        = np.sqrt(psd_raw)

fd_col     = np.fft.rfft(col_noise)
freqs_out  = np.fft.rfftfreq(N, 1.0 / FS)
asd_interp = np.interp(freqs_out, f_raw, asd_est)
asd_interp[0] = asd_interp[1]

fd_white                  = fd_col / asd_interp
fd_white[freqs_out < 20]  = 0.0

col_whitened = np.fft.irfft(fd_white, n=N)
f_wh, psd_wh = welch(col_whitened, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].loglog(f_raw[1:], np.sqrt(psd_raw[1:]), lw=2, color='C1')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('ASD')
axes[0].set_title('Step 1: Raw coloured noise\nLoud at low f, quiet at high f')
axes[0].set_xlim(20, FS // 2); axes[0].grid(True, which='both', alpha=0.3)

axes[1].loglog(f_raw[1:], asd_est[1:], lw=2, color='C3')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('ASD (noise estimate)')
axes[1].set_title('Step 2: Estimate the noise ASD\nDivide every frequency bin by this curve')
axes[1].set_xlim(20, FS // 2); axes[1].grid(True, which='both', alpha=0.3)
# Shade the area under the curve to emphasise "divide by this"
axes[1].fill_between(f_raw[1:], asd_est[1:].min() * 0.5, asd_est[1:],
                     alpha=0.15, color='C3')

mean_wh = 10 * np.log10(np.median(psd_wh[1:]))
axes[2].semilogx(f_wh[1:], 10 * np.log10(psd_wh[1:]), lw=1.5, color='C2')
axes[2].axhline(mean_wh, color='red', ls='--', lw=2, label='Mean level (flat)')
axes[2].set_xlabel('Frequency (Hz)'); axes[2].set_ylabel('Power (dB)')
axes[2].set_title('Step 3: After whitening\nFlat — every frequency equally loud')
axes[2].set_xlim(20, FS // 2); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

fig.suptitle('How whitening works: divide each frequency bin by its own noise level', fontsize=12)
plt.tight_layout(); plt.show()

### Why Sage whitens before the neural network

Raw detector strain spans **ten orders of magnitude** across frequency. A neural network
seeing those raw numbers would have to simultaneously learn which frequency bands to trust
*and* what a signal looks like. Whitening removes that burden: every frequency starts equal,
and the network's only job is to recognise the **shape** of a chirp.

---
## Demos: Hearing Whitening in Action

Three demos make whitening **audible** using everyday sounds.
Each follows the same structure:

1. Signal alone — listen and memorise how it sounds.
2. Signal buried in coloured noise — try to hear it (probably can't).
3. After whitening — the signal returns.

In [ ]:
def whiten(mixed, noise_reference, f_low=40.0):
    """Whiten `mixed` using the ASD estimated from `noise_reference`."""
    f_w, psd  = welch(noise_reference, fs=FS, nperseg=FS // 4, noverlap=FS // 8)
    asd       = np.sqrt(psd)
    fd        = np.fft.rfft(mixed)
    freqs_out = np.fft.rfftfreq(len(mixed), 1.0 / FS)
    asd_interp    = np.interp(freqs_out, f_w, asd)
    asd_interp[0] = asd_interp[1]
    fd_white              = fd / asd_interp
    fd_white[freqs_out < f_low] = 0.0
    return np.fft.irfft(fd_white, n=len(mixed))

---
### Demo 1 — A Melody Buried in Coloured Noise

A pentatonic melody (C–E–G–A–C) is hidden under 1/f² coloured noise at a **6:1 amplitude ratio**.

In the spectrogram, each note appears as a **horizontal band** at a fixed frequency.
Those bands are invisible before whitening — swamped by the bright low-frequency noise.
After whitening, the noise floor flattens and the bands become clearly visible.

In [ ]:
note_freqs = [261.63, 329.63, 392.00, 440.00, 523.25]  # C4 E4 G4 A4 C5
note_dur   = DURATION / len(note_freqs)

melody = np.zeros(N)
for i, freq in enumerate(note_freqs):
    start = int(i * note_dur * FS)
    end   = int((i + 1) * note_dur * FS)
    seg_t = np.linspace(0, note_dur, end - start, endpoint=False)
    tone  = (np.sin(2 * np.pi * freq       * seg_t)
           + 0.50 * np.sin(2 * np.pi * 2 * freq * seg_t)
           + 0.25 * np.sin(2 * np.pi * 3 * freq * seg_t)
           + 0.12 * np.sin(2 * np.pi * 4 * freq * seg_t))
    env         = np.ones(len(seg_t))
    fade        = int(0.05 * FS)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    melody[start:end] = tone * env

melody /= np.max(np.abs(melody))
melody *= 0.10

mixed_music    = col_noise + melody
whitened_music = whiten(mixed_music, col_noise)

ratio = np.sqrt(np.mean(col_noise**2)) / np.sqrt(np.mean(melody**2))
print(f'Noise is {ratio:.1f}x louder (RMS) than the melody.\n')
print('1) Melody alone:')
display(Audio(to_audio(melody), rate=FS))
print('2) Mixed — melody + coloured noise:')
display(Audio(to_audio(mixed_music), rate=FS))
print('3) After whitening:')
display(Audio(to_audio(whitened_music), rate=FS))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
specgram(axes[0, 0], melody,         'Melody alone\nHorizontal bands = note frequencies', vmin=-90)
specgram(axes[0, 1], col_noise,      'Coloured noise alone\nBright at bottom, dark at top', vmin=-90)
specgram(axes[1, 0], mixed_music,    'Mixed (melody + noise)\nBands invisible — swamped', vmin=-90)
specgram(axes[1, 1], whitened_music, 'After whitening\nBands visible — noise floor flat', vmin=-50, vmax=10)
for ax in axes.flat:
    ax.set_ylim(50, 3000)
fig.suptitle('Demo 1 — Melody: spectrograms', fontsize=13)
plt.tight_layout(); plt.show()

---
### Demo 2 — A Chirp (Gravitational-Wave Analogue)

A chirp sweeps from **80 Hz to 1200 Hz** over 5 seconds — the same rising-frequency shape
a binary black hole inspiral produces, scaled to the audio band.

Unlike the melody, a chirp has no fixed note. Its frequency rises continuously.
In the spectrogram it appears as a **diagonal line** — time on x-axis, rising frequency on
y-axis. That diagonal is the GW signal shape. Whitening makes it visible.

In [ ]:
chirp_sig = signal.chirp(t, f0=80, f1=1200, t1=DURATION, method='quadratic')
ramp      = int(0.05 * FS)
chirp_sig[:ramp]  *= np.linspace(0, 1, ramp)
chirp_sig[-ramp:] *= np.linspace(1, 0, ramp)
chirp_sig *= 0.10

mixed_chirp    = col_noise + chirp_sig
whitened_chirp = whiten(mixed_chirp, col_noise)

ratio_c = np.sqrt(np.mean(col_noise**2)) / np.sqrt(np.mean(chirp_sig**2))
print(f'Noise is {ratio_c:.1f}x louder than the chirp.\n')
print('1) Chirp alone — a rising tone:')
display(Audio(to_audio(chirp_sig), rate=FS))
print('2) Mixed — chirp + coloured noise:')
display(Audio(to_audio(mixed_chirp), rate=FS))
print('3) After whitening:')
display(Audio(to_audio(whitened_chirp), rate=FS))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
specgram(axes[0, 0], chirp_sig,      'Chirp alone\nDiagonal = rising frequency', vmin=-90)
specgram(axes[0, 1], col_noise,      'Coloured noise alone', vmin=-90)
specgram(axes[1, 0], mixed_chirp,    'Mixed (chirp + noise)\nDiagonal invisible', vmin=-90)
specgram(axes[1, 1], whitened_chirp, 'After whitening\nDiagonal clearly visible', vmin=-50, vmax=10)
for ax in axes.flat:
    ax.set_ylim(30, 2000)
fig.suptitle('Demo 2 — Chirp (gravitational-wave analogue): spectrograms', fontsize=13)
plt.tight_layout(); plt.show()

---
## Summary

| | White noise | Coloured noise | After whitening |
|---|---|---|---|
| **Power spectrum** | Flat | Steep 1/f² | Flat |
| **Sounds like** | Uniform hiss | Bass rumble | Uniform hiss |
| **Signal visible?** | — (baseline) | No | **Yes** |

Whitening converts coloured noise *into* white noise — which is exactly what the Sage neural
network expects as its background. Once the background is flat, the network needs only to ask
one question: *is there coherent structure here that random hiss would not produce?*

The melody has discrete horizontal bands; the chirp has a rising diagonal.
These are the two archetypes of signal structure. A real binary merger is a chirp.
The network's job is to find it.